# ML_G0_P0002_REVIEW_answer

## 0. 정답본 범위
- Gate: G0
- Phase: P0002 REVIEW
- Topic: DataFrame vs Series vs ndarray vs tensor
- Problem notebook: ML_G0_P0002_REVIEW.ipynb
- Correct answers included: Yes
- Output language: Korean
- Data directory: data/ML_G0_P0002

## 1. 핵심 기준표

| 개념 | 기준 |
|---|---|
| DataFrame | column/index/dtype/결측치/EDA를 통한 의미 확인용 표 구조 |
| Series | 1차원 labeled array, 보통 y 한 열로 사용 |
| ndarray | column 이름 없이 값과 shape만 남은 NumPy 계산 배열 |
| tensor | 딥러닝 프레임워크에서 batch/GPU/자동미분과 연결되는 n차원 수치 배열 |
| Flatten | shape 변환 |
| normalization | value scale 변환 |

## 2. 데이터 확인 코드


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data/ML_G0_P0002")
assert DATA_DIR.exists()

students = pd.read_csv(DATA_DIR / "students_scores.csv")
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")
traffic = pd.read_csv(DATA_DIR / "traffic_congestion.csv")
images = np.load(DATA_DIR / "mock_images.npz")

assert students.shape == (8, 6)
assert customers.shape == (8, 8)
assert traffic.shape == (8, 8)
assert images["images_gray"].shape == (12, 28, 28)
assert images["images_flat"].shape == (12, 784)
assert images["labels"].shape == (12,)

print("review data ok")


## R-1. 네 자료구조의 역할 압축 비교

### 정답
DataFrame은 column 이름, index, dtype, 결측치 상태를 함께 보존하는 2차원 표 구조이며 사람이 데이터 의미를 확인하고 EDA와 전처리를 설계하기 좋다. Series는 1차원 labeled array로 DataFrame의 한 column 또는 y target으로 자주 사용된다. ndarray는 column 이름 없이 값과 shape 중심으로 남은 NumPy 수치 배열이며 계산에 적합하다. tensor는 딥러닝 프레임워크에서 batch 연산, GPU, 자동미분과 연결되는 n차원 수치 배열이다.

### 근거
DataFrame/Series는 의미와 label을 보존하고, ndarray/tensor는 모델 계산을 위한 수치 구조에 가깝다.

### 자주 하는 오답
Series를 행렬로 착각한다. DataFrame을 모델이 의미까지 이해한다고 생각한다. tensor를 단순히 이미지 전용 구조라고 생각한다.

### 오답튜터 기준
Series를 1차원 labeled array로 고정해 설명하게 한다. tensor는 ndarray와 비슷하지만 딥러닝 프레임워크의 batch/GPU/자동미분 맥락과 연결된다고 교정한다.

### 최종 답안형
DataFrame은 의미 확인용 표, Series는 1차원 labeled target/column, ndarray는 이름 없는 계산 배열, tensor는 딥러닝 학습용 n차원 수치 구조다.


## R-2. customer_purchase.csv에서 dtype과 X/y 판단

### 정답
`purchased`가 y target이다. X 후보는 `age`, `income`, `visit_count`, `cart_amount`, 그리고 encoding 후의 `city`, `device_type`이다. `customer_id`는 식별자이므로 feature에서 제외하는 것이 안전하다. `city`, `device_type`은 object/string dtype이므로 바로 모델에 넣기 어렵고, one-hot encoding 같은 categorical 처리가 필요하다. 문제 유형은 구매 여부 `0/1`을 예측하는 이진분류다.

### 근거
숫자 column이라고 모두 feature는 아니며, ID와 target은 분리해야 한다. 문자열 category는 모델 입력 전 수치화해야 한다.

### 자주 하는 오답
`customer_id`를 숫자라서 feature에 넣는다. `purchased`를 X에 포함한다. `city`, `device_type`을 `to_numpy()`만 하면 자동 처리된다고 본다.

### 오답튜터 기준
ID는 식별자, target은 y, object column은 encoding 대상으로 분리하게 한다.

### 최종 답안형
X는 `age`, `income`, `visit_count`, `cart_amount`와 encoding된 `city/device_type`, y는 `purchased`, `customer_id`는 제외한다. 이 문제는 이진분류다.


## R-3. to_numpy() 이후 남는 것과 사라지는 것

### 정답
`X_df`에는 column 이름, index, dtype, feature 의미가 있다. `X_np`에는 값, shape, dtype만 남고 column 이름과 index, feature 의미는 사라진다. `to_numpy()`는 현재 column 순서대로 값을 배열에 넣기 때문에 feature 순서 자체는 유지된다. 하지만 이름이 사라지므로 `feature_cols` 순서를 따로 기록해야 한다. inference 때 순서가 바뀌면 모델은 다른 feature 값을 같은 위치의 입력으로 오해해 잘못 예측할 수 있다.

### 근거
to_numpy 이후 사라지는 것은 순서가 아니라 이름과 의미다. 순서는 배열 위치로 남지만, 그 위치가 무엇인지 metadata 없이는 알기 어렵다.

### 자주 하는 오답
to_numpy 후 column 이름도 남는다고 생각한다. 또는 feature 순서까지 완전히 사라진다고 혼동한다.

### 오답튜터 기준
“순서는 유지, 이름은 소실”을 분리해 설명하게 한다.

### 최종 답안형
`X_np`에는 값과 shape만 남으므로 변환 전 `feature_cols` 순서를 반드시 기록해야 한다.


## R-4. 이미지 tensor, Flatten, normalization 구분

### 정답
12는 class id가 아니라 이미지 sample 개수다. 28x28은 각 grayscale 이미지의 세로와 가로 pixel 구조다. 784는 `28*28`을 1차원으로 펼친 feature 개수다. labels는 각 이미지에 대응되는 target class label이다. Flatten은 `(12, 28, 28)`을 `(12, 784)`처럼 shape만 바꾸는 작업이다. normalization은 pixel 값 `0~255` 같은 value scale을 `0~1` 등으로 바꾸는 작업이다. Flatten하면 2차원 공간 구조와 인접 pixel 관계가 사라진다.

### 근거
Flatten은 shape 변환이고 normalization은 값 범위 변환이다. 0~255는 차원이 아니라 pixel 값의 범위다.

### 자주 하는 오답
12를 class 수로 착각한다. 0~255를 차원으로 착각한다. Flatten과 normalization을 같은 처리로 설명한다.

### 오답튜터 기준
shape 축의 의미와 value scale의 의미를 따로 묻는다.

### 최종 답안형
이미지에서 12는 sample 수, 28x28은 공간 구조, 784는 flatten된 feature 수이며, Flatten은 shape 변환이고 normalization은 값 스케일 변환이다.


## R-5. 전체 파이프라인 한 번에 설명하기

### 정답
CSV/raw data는 아직 의미 확인과 전처리가 필요한 원천 데이터다. DataFrame 단계에서는 column 이름, dtype, 결측치, category, target 후보를 확인한다. `head()`는 샘플과 column 구조, `info()`는 dtype/non-null/object column, `describe()`는 수치 분포를 본다. X/y 분리는 무엇으로 무엇을 예측할지 정하는 단계다. categorical/결측 처리는 column 의미와 dtype을 알아야 하므로 DataFrame 단계에서 설계하는 것이 자연스럽다. ndarray/tensor 변환은 모델이 빠르게 수치 계산을 하도록 만드는 단계다. `model.fit(X_train, y_train)`은 X를 보고 y를 맞히도록 모델 파라미터를 학습하는 과정이다.

### 근거
의미 확인 단계와 계산 단계가 다르다. DataFrame은 해석/전처리 설계, ndarray/tensor는 계산/학습에 가깝다.

### 자주 하는 오답
DataFrame을 곧바로 모델이 의미까지 이해한다고 생각한다. X/y 분리 전에 ndarray로 먼저 바꾼다. categorical 처리를 생략한다.

### 오답튜터 기준
각 단계의 산출물과 책임을 한 문장씩 말하게 한다.

### 최종 답안형
DataFrame으로 의미와 전처리를 설계하고, X/y를 정리한 뒤 ndarray/tensor로 바꿔 `model.fit`에서 수치 학습을 수행한다.


## 3. 채점 기준

| 기준 | 확인할 것 |
|---|---|
| 자료구조 구분 | DataFrame/Series/ndarray/tensor의 역할을 분리하는가 |
| shape 해석 | sample 축, feature 축, target shape를 구분하는가 |
| 정보 손실 | `to_numpy()` 이후 이름/의미가 사라짐을 설명하는가 |
| dtype 처리 | object/categorical column을 인코딩 대상으로 보는가 |
| 이미지 처리 | Flatten과 normalization을 구분하는가 |
| 모델 입력 | `model.fit(X_train, y_train)`의 X/y 대응을 설명하는가 |

## 4. 오답튜터 기준표

1. Series를 행렬로 착각
2. DataFrame을 모델이 의미까지 이해한다고 착각
3. ndarray에 column 이름이 보존된다고 착각
4. object/string dtype을 그대로 모델에 넣으려 함
5. `customer_id` 같은 식별자를 의미 있는 feature로 착각
6. Flatten과 normalization을 같은 처리로 착각
7. `0~255`를 차원이 아니라 값 범위로 보지 못함
8. `model.fit`을 단순 저장이나 실행 호출로만 이해

## 5. 다음 학습 연결

- P0002 본편의 오염된 답안은 정답본과 분리해서 관리한다.
- 다음 단계에서는 categorical encoding, train/test split, scaling, leakage 방지를 더 깊게 점검한다.


<!-- AGC:REVIEW_GRADING_NOTES_START P0002_REVIEW -->

## 6. P0002 REVIEW 채점 반영 및 유의사항

### 6-1. 작업 기준

이번 섹션은 새 문제를 생성하지 않고, 사용자가 작성한 `ML_G0_P0002_REVIEW.ipynb`를 기준으로 채점, 오답 원인 분석, 정답형 피드백, 최종 정리, 다음 강 준비만 통합한 것이다.

- 현재 노드: Gate 0 / P0002 REVIEW - DataFrame vs Series vs ndarray vs tensor
- 후속 노드: P0003 - shape/axis
- 관련 강의 축: 3강 NumPy의 `ndarray/shape/dtype`, 4강 Pandas의 `Series/DataFrame`, 14강의 `DataFrame -> X/y -> model-ready tensor`, 15강의 image tensor / Flatten 연결
- 보존 원칙: 풀이 노트북 `ML_G0_P0002_REVIEW.ipynb`는 수정하지 않고, 채점과 유의사항은 이 정답본에만 기록한다.

### 6-2. 전체 판정

| 문항 | 점수 | 판정 |
|---|---:|---|
| R-1 | 80 | 핵심 구분은 잡힘. tensor와 Series 설명 일부 보정 필요 |
| R-2 | 72 | X/y 판단은 좋음. 표기와 문제 유형 명시 부족 |
| R-3 | 82 | `to_numpy()` 핵심 이해 양호. 답안이 코드 출력 중심으로 산만함 |
| R-4 | 78 | Flatten/normalization 구분은 잡힘. 이미지 axis 표현 일부 오류 |
| R-5 | 62 | 전체 파이프라인 이해는 있으나 답안형이 너무 압축됨 |
| 종합 | 75/100 | 부분 통과. P0002 핵심은 잡았고, 최종 답안형 정리가 필요 |

현재 상태는 개념 이해는 통과권이지만, 시험 답안형과 노트북 답안형은 아직 다듬어야 하는 상태다. 다음 강으로 넘어갈 수는 있으나, P0002 최종 문장 몇 개는 고정하고 가는 것이 좋다.

### 6-3. R-1 유의사항: 네 자료구조 비교

잘한 점:
- DataFrame을 사람이 보고 전처리, 기술통계, dtype, 결측치, X/y 점검을 하는 구조로 잡은 것은 정확하다.
- Series를 행렬이 아니라 벡터에 가까운 1차원 labeled array로 고친 것도 좋다.

교정할 점:
- ndarray에서 index label과 column name은 사라지지만, 위치 기반 indexing 자체는 남는다.
- tensor는 이미지 전용 구조가 아니다. tabular 데이터도 딥러닝에 들어가면 `(N, p)` 형태의 2D tensor가 된다.

정답형 피드백:

```text
DataFrame은 column 이름, index, dtype, 결측치, 범주형 여부를 확인하며 사람이 데이터 의미를 해석하기 위한 2차원 표 구조다.
Series는 한 column 또는 target y를 담는 1차원 labeled array다.
ndarray는 column 이름 없이 값과 shape만 가진 NumPy 수치 배열이며, 위치 기반 인덱싱과 빠른 계산에 적합하다.
tensor는 딥러닝 프레임워크에서 batch 연산, GPU 연산, 자동미분과 연결되는 n차원 수치 배열이다.
```

오답 원인:
- Series, ndarray, tensor를 모두 배열로 묶어 이해하는 방향은 맞지만, 레이블이 있는 배열인지, 순수 값 배열인지, 딥러닝 계산 그래프와 연결되는 배열인지 더 분리해야 한다.

### 6-4. R-2 유의사항: `customer_purchase.csv`의 X/y와 dtype

잘한 점:
- `customer_id`와 `purchased`를 제외하고 feature 후보를 잡은 것은 맞다.
- `city`, `device_type`이 문자열/범주형이라 바로 모델에 넣기 어렵고 one-hot encoding이 필요하다고 본 것도 정확하다.

교정할 점:
- `y column = list['purchased']`가 아니라 `y column = purchased`라고 써야 한다.
- `feature에서 제외할 column`에는 코드 전체가 아니라 `customer_id, purchased`를 쓴다.
- `purchased`는 0/1 구매 여부이므로 문제 유형은 이진분류다.
- `print(list[feature_cols])`, `print(list["purchased"])`는 의도한 출력이 아니다. 실제 리스트를 보려면 `print(feature_cols)`를 사용한다.

정답형 피드백:

```text
X column 후보 = age, income, city, device_type, visit_count, cart_amount
y column = purchased
feature에서 제외할 column = customer_id, purchased
object/string dtype이라 바로 모델에 넣기 어려운 column = city, device_type
필요한 전처리 = city와 device_type은 순서 없는 범주형 feature이므로 one-hot encoding 등으로 숫자화해야 한다.
문제 유형 = purchased가 0/1 구매 여부이므로 이진분류다.
```

코드 확인 예시:

```python
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")
feature_cols = ["age", "income", "city", "device_type", "visit_count", "cart_amount"]
target_col = "purchased"

print(feature_cols)
print(target_col)
print(customers[feature_cols].dtypes)
```

오답 원인:
- 개념은 맞지만 코드 표현을 답안 문장처럼 쓰는 습관이 있다. 채점 답안에는 코드 조각보다 어떤 column이 무엇인지 명확히 써야 한다.

### 6-5. R-3 유의사항: `to_numpy()` 이후 정보 손실

잘한 점:
- feature 순서는 유지된다.
- column 이름은 사라진다.
- 순서를 기록하지 않으면 inference 때 feature가 엉킬 수 있다는 핵심을 잡았다.

정답형 피드백:

```text
X_df에 있고 X_np에 없는 것 = column 이름, index, feature 의미
X_np에 남는 것 = 실제 값, shape, dtype, column 순서
feature 순서는 유지되는가 = 유지된다. X_df의 column 순서대로 ndarray 열이 만들어진다.
feature 순서를 기록해야 하는 이유 = ndarray는 column 이름을 기억하지 않기 때문에, 각 열이 어떤 feature인지 사람이 따로 관리해야 한다.
잘못된 순서로 inference하면 생기는 문제 = 학습 때 age, income, visit_count 순서로 학습했는데 예측 때 income, age, visit_count 순서로 넣으면 모델이 각 값을 잘못된 feature로 해석해 예측이 망가진다.
```

코드 확인 예시:

```python
students = pd.read_csv(DATA_DIR / "students_scores.csv")
feature_cols = ["study_hours", "attendance_rate", "assignment_score", "midterm_score"]
X_df = students[feature_cols]
X_np = X_df.to_numpy()

print(X_df.columns.tolist())
print(X_np.shape)
print(X_np.dtype)
```

오답 원인:
- 사고는 맞다. 다만 code cell은 확인용, Markdown은 판단용으로 분리해야 한다.

### 6-6. R-4 유의사항: image tensor, Flatten, normalization

잘한 점:
- `12 = 이미지 샘플 수`, `784 = 28 x 28을 flatten한 길이`, `normalization = 값 범위 변환`, `Flatten = 차원/shape 변환`을 구분한 것은 맞다.

교정할 점:
- `(12, 28, 28)`에서 axis 해석은 다음처럼 고정한다.

```text
axis 0 = 이미지/sample 축 = 12
axis 1 = 높이 = 28
axis 2 = 너비 = 28
```

- `labels`는 DataFrame column 이름이 아니라 각 이미지에 대응하는 target array다.
- Flatten하면 사라지는 것은 2D 공간 구조, 인접 픽셀 관계, 위/아래/좌/우 위치 관계다.

정답형 피드백:

```text
12의 의미 = 이미지 샘플 12개다. class id가 아니다.
28 x 28의 의미 = 이미지 한 장이 세로 28픽셀, 가로 28픽셀의 2D grid라는 뜻이다.
784의 의미 = 28 x 28 픽셀을 한 줄로 펼친 feature 개수다.
labels = 각 이미지에 대응하는 target label 배열이다. shape (12,)는 이미지 12장에 정답 label이 12개 있다는 뜻이다.
Flatten의 의미 = (28, 28) 이미지 grid를 (784,) 1차원 벡터로 펴는 shape 변환이다.
normalization과 Flatten의 차이 = normalization은 0~255 값을 0~1 같은 범위로 바꾸는 value scale 변환이고, Flatten은 배열 모양을 바꾸는 shape 변환이다.
Flatten하면 사라지는 정보 = 픽셀의 2D 공간 배치, 인접 관계, local pattern 구조가 사라진다.
```

코드 확인 예시:

```python
images = np.load(DATA_DIR / "mock_images.npz")
print(images["images_gray"].shape)  # (12, 28, 28)
print(images["images_flat"].shape)  # (12, 784)
print(images["labels"].shape)       # (12,)
```

오답 원인:
- 이미지의 axis와 value는 구분하기 시작했지만, axis 번호를 실제 shape 위치와 정확히 연결하는 훈련이 더 필요하다. 이 부분은 P0003 shape/axis에서 바로 이어진다.

### 6-7. R-5 유의사항: 전체 파이프라인 설명

가장 약한 문항이다. 방향은 맞지만, 최종 답안형으로는 너무 짧다. 특히 `최종 한 문장 요약 = 모델링`은 부족하다.

정답형 피드백:

```text
DataFrame 단계의 역할 = column 이름, dtype, 결측치, 범주형 여부, target 위치를 확인하며 사람이 데이터 의미를 해석하고 전처리 방향을 정하는 단계다.
head/info/describe의 역할 = head는 앞부분 샘플과 column 구조 확인, info는 dtype과 non-null count 확인, describe는 수치형 column의 분포와 scale 확인이다.
X/y 분리의 이유 = 모델이 볼 입력 feature와 맞혀야 할 target을 분리하고, target leakage를 막기 위해서다.
categorical/결측 처리를 DataFrame 단계에서 하는 이유 = column 이름과 dtype이 남아 있어 어떤 열을 어떻게 처리해야 하는지 판단하기 쉽기 때문이다. ndarray로 변환하면 column 의미가 사라져 처리 오류가 생기기 쉽다.
ndarray/tensor로 바꾸는 이유 = 모델은 결국 수치 계산을 수행하므로, 전처리된 데이터를 계산 가능한 숫자 배열로 넘겨야 하기 때문이다.
model.fit(X_train, y_train)의 의미 = X_train을 보고 y_train을 맞히도록 모델 파라미터를 학습시키는 과정이다.
최종 한 문장 요약 = DataFrame에서 의미와 전처리를 확정하고, ndarray/tensor로 계산 가능한 형태를 만든 뒤, model.fit으로 X를 y에 맞추도록 학습시킨다.
```

오답 원인:
- 전체 흐름을 이해는 했지만, 시험 답안형 문장으로 압축하는 능력이 아직 약하다. 한 단어로 끝내면 채점자가 무엇을 아는지 확인할 수 없다.

### 6-8. 코드/노트북 운영 유의사항

이번 리뷰 노트북은 이전보다 훨씬 깨끗하다. 다만 아래 원칙을 고정한다.

```text
code cell = 데이터 확인
Markdown cell = 판단과 설명
answer notebook = 정답, 해설, 오답튜터, 최종 답안형
solved notebook = 사용자 원답 보존
```

유의할 점:
- code cell에 너무 많은 정답형 설명을 주석으로 쓰지 않는다.
- code cell에는 실행 가능한 확인 코드만 둔다.
- Markdown 답안에는 코드 출력이 아니라 판단 문장을 쓴다.
- `print(list[feature_cols])`처럼 타입 힌트 문법으로 보이는 표현을 피하고, 실제 객체를 출력한다.
- `to_numpy()` 이후에는 값과 shape는 남지만 column name과 feature 의미는 사라진다는 문장을 고정한다.

### 6-9. 가상환경 실행 및 저장 유의사항

이번 노트북은 `.venv` 기준으로 검증한다. 시스템 Python에 `nbformat`이 없을 수 있으므로, 노트북 검증과 실행 확인은 가능하면 프로젝트 가상환경을 우선 사용한다.

권장 확인 명령:

```bash
.venv/bin/python -V
.venv/bin/python -m pip show nbformat
.venv/bin/python - <<'PY_CHECK'
import json
from pathlib import Path
for path in ["ML_G0_P0002_REVIEW.ipynb", "ML_G0_P0002_REVIEW_answer.ipynb"]:
    json.loads(Path(path).read_text(encoding="utf-8"))
    print(path, "json ok")
PY_CHECK
```

데이터 import 확인:

```python
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data/ML_G0_P0002")
assert DATA_DIR.exists()

students = pd.read_csv(DATA_DIR / "students_scores.csv")
customers = pd.read_csv(DATA_DIR / "customer_purchase.csv")
traffic = pd.read_csv(DATA_DIR / "traffic_congestion.csv")
images = np.load(DATA_DIR / "mock_images.npz")

assert students.shape == (8, 6)
assert customers.shape == (8, 8)
assert traffic.shape == (8, 8)
assert images["images_gray"].shape == (12, 28, 28)
assert images["images_flat"].shape == (12, 784)
assert images["labels"].shape == (12,)
```

저장/커밋 전 체크:

```bash
git status --short --branch
python3 -m json.tool ML_G0_P0002_REVIEW.ipynb > /dev/null
python3 -m json.tool ML_G0_P0002_REVIEW_answer.ipynb > /dev/null
```

주의:
- Jupyter나 VSCode에서 열린 버퍼가 저장되지 않았을 수 있으므로, 채점 기준은 디스크에 저장된 `.ipynb` 파일이다.
- 풀이 노트북을 자동 정리하거나 덮어쓰지 않는다.
- 정답/채점/오답노트는 `_answer.ipynb`에만 누적한다.

### 6-10. 다음 강 준비: P0003 shape/axis

P0003으로 넘어가기 전에 아래 문장들을 고정한다.

```text
Series y = shape (n,)
DataFrame 한 열 y = shape (n, 1)
DataFrame은 column 의미를 보존한다.
ndarray는 column 이름 없이 값과 shape 중심이다.
axis 0은 보통 sample/batch 축이다.
이미지 (N, H, W)에서 N은 sample 수, H/W는 공간 축이다.
Flatten은 shape 변환이고 normalization은 value scale 변환이다.
```

<!-- AGC:REVIEW_GRADING_NOTES_END P0002_REVIEW -->
